# Equipment Request API — Consumer Demo Notebook

This notebook demonstrates how to consume the Equipment Request API built on FastAPI and backed by Microsoft Dataverse.

It covers:

- Calling the API
- Retrieving normalized Dataverse data
- Performing basic analytics
- Creating, updating, and deleting records
- Showing how downstream systems (Fabric, Databricks, Power BI) would use this API

This notebook acts as a portfolio‑ready demonstration of the API’s practical value.

In [1]:
# ---------------------------------------------------------
# Cell 2 — Setup: Imports and API Base URL
# ---------------------------------------------------------
# This cell initializes the environment for interacting with
# the Equipment Request API. It imports required libraries
# and defines the base URL for all API calls.
# ---------------------------------------------------------

import requests
from datetime import datetime, timedelta
from collections import Counter

BASE_URL = "http://127.0.0.1:8000"

print("API Base URL:", BASE_URL)

API Base URL: http://127.0.0.1:8000


In [2]:
# ---------------------------------------------------------
# Cell 3 — Retrieve All Equipment Requests
# ---------------------------------------------------------
# Calls GET /equipment-requests/ to retrieve all normalized
# Dataverse records. This is the primary entry point for
# downstream analytics and integrations.
# ---------------------------------------------------------

resp = requests.get(f"{BASE_URL}/equipment-requests/")
resp.raise_for_status()

records = resp.json()
print(f"Retrieved {len(records)} equipment requests.")
records[:3]  # preview first 3

Retrieved 16 equipment requests.


[{'id': '8df46aee-d30a-f111-8406-6045bd0a812b',
  'approval_stage': 519970002,
  'equipment_type': 519970002,
  'estimated_cost': 75.0,
  'justification': 'The HDMI port stopped working...',
  'quantity': 1,
  'request_date': '2026-02-16',
  'requested_by': 'mlloyd9@pods9.onmicrosoft.com',
  'status': 519970000,
  'is_archived': False,
  'notes': None,
  'new_column': None,
  'created_on': '2026-02-16T01:07:58Z',
  'modified_on': '2026-02-16T05:08:36Z',
  'owner_id': '203aa3b7-defe-f011-8406-000d3a306f55'},
 {'id': '73dddc50-d40a-f111-8406-6045bd0a812b',
  'approval_stage': 519970002,
  'equipment_type': 519970008,
  'estimated_cost': 1000.0,
  'justification': 'Lamp for lighting during online meetings.',
  'quantity': 1,
  'request_date': '2026-02-16',
  'requested_by': 'mlloyd9@pods9.onmicrosoft.com',
  'status': 519970000,
  'is_archived': False,
  'notes': 'Lamp for lighting during online meetings.',
  'new_column': None,
  'created_on': '2026-02-16T01:10:44Z',
  'modified_on': '20

In [3]:
# ---------------------------------------------------------
# Cell 4 — Analysis: Count Requests by Person
# ---------------------------------------------------------
# Demonstrates how downstream systems (Fabric, Databricks,
# Power BI) can easily analyze normalized API data.
# ---------------------------------------------------------

requested_by_counts = Counter(
    r["requested_by"] for r in records if r["requested_by"]
)

requested_by_counts

Counter({'mlloyd9@pods9.onmicrosoft.com': 16})

In [4]:
# ---------------------------------------------------------
# Cell 5 — Analysis: Total Estimated Cost of Pending Requests
# ---------------------------------------------------------
# Shows how business metrics can be computed directly from
# the API output without needing Dataverse knowledge.
# ---------------------------------------------------------

pending_cost = sum(
    r["estimated_cost"] or 0
    for r in records
    if r["status"] == "Pending"
)

print(f"Total estimated cost of pending requests: ${pending_cost:,.2f}")

Total estimated cost of pending requests: $0.00


In [5]:
# ---------------------------------------------------------
# Cell 6 — Analysis: Identify Old Requests
# ---------------------------------------------------------
# Demonstrates date-based filtering and time-window analysis.
# Useful for SLA monitoring, backlog cleanup, and reporting.
# ---------------------------------------------------------

cutoff = datetime.utcnow() - timedelta(days=30)

old_requests = [
    r for r in records
    if r["created_on"] 
    and datetime.fromisoformat(r["created_on"].replace("Z", "")) < cutoff
]

print(f"Found {len(old_requests)} requests older than 30 days.")
old_requests[:3]

Found 16 requests older than 30 days.


C:\Users\mrllo\AppData\Local\Temp\ipykernel_14232\3715157293.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  cutoff = datetime.utcnow() - timedelta(days=30)


[{'id': '8df46aee-d30a-f111-8406-6045bd0a812b',
  'approval_stage': 519970002,
  'equipment_type': 519970002,
  'estimated_cost': 75.0,
  'justification': 'The HDMI port stopped working...',
  'quantity': 1,
  'request_date': '2026-02-16',
  'requested_by': 'mlloyd9@pods9.onmicrosoft.com',
  'status': 519970000,
  'is_archived': False,
  'notes': None,
  'new_column': None,
  'created_on': '2026-02-16T01:07:58Z',
  'modified_on': '2026-02-16T05:08:36Z',
  'owner_id': '203aa3b7-defe-f011-8406-000d3a306f55'},
 {'id': '73dddc50-d40a-f111-8406-6045bd0a812b',
  'approval_stage': 519970002,
  'equipment_type': 519970008,
  'estimated_cost': 1000.0,
  'justification': 'Lamp for lighting during online meetings.',
  'quantity': 1,
  'request_date': '2026-02-16',
  'requested_by': 'mlloyd9@pods9.onmicrosoft.com',
  'status': 519970000,
  'is_archived': False,
  'notes': 'Lamp for lighting during online meetings.',
  'new_column': None,
  'created_on': '2026-02-16T01:10:44Z',
  'modified_on': '20

In [ ]:
# ---------------------------------------------------------
# Cell 7 — Create a New Equipment Request
# ---------------------------------------------------------
# Calls POST /equipment-requests to create a new Dataverse
# record through the API. Demonstrates write operations.
# ---------------------------------------------------------

new_request = {
    "approval_stage": 1,
    "equipment_type": 2,
    "estimated_cost": 2200,
    "justification": "New workstation for analytics",
    "quantity": 1,
    "requested_by": "Michael Lloyd",
    "status": 1,
    "notes": "Needed for Fabric + Snowflake development"
}

resp = requests.post(f"{BASE_URL}/equipment-requests", json=new_request)
resp.raise_for_status()

new_id = resp.json()
new_id

In [ ]:
# ---------------------------------------------------------
# Cell 8 — Retrieve Newly Created Record
# ---------------------------------------------------------
# Calls GET /equipment-requests/{id} to verify that the
# record was successfully created and normalized.
# ---------------------------------------------------------

resp = requests.get(f"{BASE_URL}/equipment-requests/{new_id}")
resp.raise_for_status()

resp.json()

In [ ]:
# ---------------------------------------------------------
# Cell 9 — Update an Existing Record
# ---------------------------------------------------------
# Calls PATCH /equipment-requests/{id} to modify fields.
# Demonstrates partial updates and Dataverse write behavior.
# ---------------------------------------------------------

update_payload = {
    "status": 2,  # Approved
    "notes": "Approved by IT"
}

resp = requests.patch(f"{BASE_URL}/equipment-requests/{new_id}", json=update_payload)
resp.raise_for_status()

resp.json()

In [ ]:
# ---------------------------------------------------------
# Cell 10 — Delete a Record
# ---------------------------------------------------------
# Calls DELETE /equipment-requests/{id} to remove or archive
# a Dataverse record. Demonstrates full CRUD lifecycle.
# ---------------------------------------------------------

resp = requests.delete(f"{BASE_URL}/equipment-requests/{new_id}")
resp.raise_for_status()

resp.json()

# Summary

This notebook demonstrated:

- How to call the Equipment Request API
- How to retrieve normalized Dataverse data
- How to perform analytics on the returned dataset
- How to create, update, and delete Dataverse records through the API

This workflow mirrors how Fabric notebooks, Databricks, Power Apps, and external systems would integrate with the API in real-world scenarios.

This notebook can be included in your portfolio as a practical demonstration of the API’s value.